# Start

Discover CoRE Stack datasets, make your first API request and save data to explore in a map or spreadsheet.

Run the cells in order. Change the place, identifier or columns to explore other records. Downloads from GeoLibre use your selected tehsil; these templates start with Hilsa, Nalanda, Bihar.


## Set up Python

Run the collapsed setup cells. They import the libraries and define `read_json`, a small response reader. It reads JSON text, treats non-standard `NaN` and `Infinity` numbers as missing, and also accepts JSON returned inside a string. HTTP errors and malformed responses remain visible. Expand the cells to read the code.


In [ ]:
import sys
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["geopandas", "matplotlib", "requests", "pyodide-http"])
    import pyodide_http
    pyodide_http.patch_all()

import os
import re
import ast
import json
from getpass import getpass
from inspect import isawaitable
from urllib.parse import urljoin
import requests
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, FileLink
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})


In [ ]:
SCOPE = json.loads("{\"state\": \"Bihar\", \"district\": \"Nalanda\", \"tehsil\": \"Hilsa\"}")
API_URL = 'https://geoserver.core-stack.org/api/v1/'
STAC_URL = 'https://spatio-temporal-asset-catalog.s3.ap-south-1.amazonaws.com/CorestackCatalogs_merged_collection/tehsil_wise/catalog.json'
YEARS = list(range(2017, 2025))


In [ ]:
"""Small response reader embedded in the notebooks' collapsed setup cell."""
import json


def read_json(response):
    """Read JSON text; represent non-standard NaN/Infinity values as missing."""
    response.raise_for_status()
    raw_text = response.text.lstrip("\ufeff")
    try:
        # Some API tables contain bare NaN or Infinity, which are not JSON numbers.
        data = json.loads(raw_text, parse_constant=lambda value: None)
        # Also accept a JSON document returned as a JSON-encoded string.
        if isinstance(data, str):
            data = json.loads(data.lstrip("\ufeff"), parse_constant=lambda value: None)
        return data
    except ValueError as error:
        raise ValueError(
            "The server response is not readable JSON. "
            "Inspect response.status_code and response.text[:500], then retry the request."
        ) from error


## Choose the place and set your API key

Edit `SCOPE` in the setup cell to change the place. The [public API guide](https://docs.core-stack.org/use-precomputed-data/public-apis/) explains registration and API keys. This cell reuses `CORE_STACK_API_KEY` or asks for it privately, then stores it in this kernel’s environment. The key is sent only to the API, in the `X-API-Key` header. Restart the kernel and run from the top after changing places.


In [ ]:
place = {key: re.sub(r"[\s_]+", "_", SCOPE[key].replace("(", "").replace(")", "")).strip("_").lower()
         for key in ["state", "district", "tehsil"]}
state, district, tehsil = place["state"], place["district"], place["tehsil"]
api_key = os.environ.get("CORE_STACK_API_KEY", "").strip()
if not api_key:
    api_key = getpass("CoRE Stack API key: ")
    if isawaitable(api_key):
        api_key = await api_key
os.environ["CORE_STACK_API_KEY"] = str(api_key).strip()
api_headers = {"X-API-Key": os.environ["CORE_STACK_API_KEY"]}
display(place)


## See the available APIs

Read the specifications at [api-doc.core-stack.org](https://api-doc.core-stack.org). This public OpenAPI document lists the GET APIs and their required parameters without an API key.


In [ ]:
response = requests.get("https://geoserver.core-stack.org/?format=openapi", timeout=90)
specification = read_json(response)
operations = {path: details["get"] for path, details in specification["paths"].items()
              if "get" in details and path.startswith("/get_")}
display(pd.DataFrame([{"API": path, "Purpose": operation.get("summary", ""),
                       "Required parameters": ", ".join(p["name"] for p in operation.get("parameters", []) if p.get("required"))}
                      for path, operation in operations.items()]))


## Choose an API and inspect its schema

Change `api_path` to one of the paths above. Parameter names, descriptions and response definitions come from the public specification. A `$ref` points to a named definition, included below.


In [ ]:
api_path = "/get_active_locations/"
operation = operations[api_path]
display(pd.DataFrame(operation.get("parameters", [])).reindex(columns=["name", "required", "type", "description"]))
display(operation.get("responses", {}))
response_schema = operation.get("responses", {}).get("200", {}).get("schema", {})
references = re.findall(r'#/definitions/([^" ]+)', json.dumps(response_schema))
display({name: specification.get("definitions", {}).get(name) for name in references})


## Make the request

Use `{}` for active locations or `place` for a tehsil API. Add `mws_id`, `uid`, or coordinates when the selected API requires them. The response stays in `api_result` for further exploration.


In [ ]:
parameters = {}  # For get_tehsil_data, use: parameters = place
response = requests.get(API_URL + api_path.lstrip("/"), params=parameters, headers=api_headers, timeout=180)
api_result = read_json(response)
display(pd.json_normalize(api_result).head() if isinstance(api_result, list) else api_result)


## Discover data and descriptions in STAC

STAC lists published datasets, field descriptions, downloads and styles. Change `dataset` to another item from the collection. Asset links are used as published, wherever the files are hosted. STAC describes asset fields; API tables may use different names and units, which are shown explicitly in the examples below.


In [ ]:
collection_url = urljoin(STAC_URL, f"{state}/{district}/{tehsil}/collection.json")
response = requests.get(collection_url, timeout=90)
collection = read_json(response)
items = pd.DataFrame([{"Item": link["href"].split("/")[-1].removesuffix(".json"),
                       "URL": urljoin(collection_url, link["href"])}
                      for link in collection["links"] if link["rel"] == "item"], columns=["Item", "URL"])
display(items)
dataset = "terrain_vector"
matches = items.loc[items["Item"].str.endswith("_" + dataset)]
item = None
if not matches.empty:
    item_url = matches.iloc[0]["URL"]
    response = requests.get(item_url, timeout=90)
    item = read_json(response)
    display(Markdown(item["properties"].get("description", "No description published.")))
    field_notes = pd.DataFrame(item["properties"].get("table:columns", []))
    display(field_notes.reindex(columns=["name", "type", "description"]).head(12))
    print("Published field count:", len(field_notes), "— use field_notes to see them all.")
    display(pd.DataFrame(item["assets"]).T.reindex(columns=["title", "type", "href"]))
else:
    print("This dataset is not listed in the tehsil's STAC collection. Available items:")
    display(items)


## Read and save micro-watershed data

The geometry API supplies boundaries; `get_tehsil_data` supplies attributes. Join them on `uid`, inspect the first record and save GeoJSON and CSV. Open the GeoJSON in QGIS or GeoLibre to explore its fields.


In [ ]:
response = requests.get(API_URL + "get_mws_geometries/", params=place, headers=api_headers, timeout=180)
mws = gpd.GeoDataFrame.from_features(read_json(response)["features"], crs="EPSG:4326")
response = requests.get(API_URL + "get_tehsil_data/", params=place, headers=api_headers, timeout=180)
api_data = read_json(response)
attributes = pd.DataFrame(api_data["mws"])
mws["uid"], attributes["uid"] = mws["uid"].astype(str), attributes["uid"].astype(str)
mws = mws.merge(attributes, on="uid", how="left", validate="one_to_one")
display(mws.drop(columns="geometry").iloc[0].to_frame("First MWS"))
mws.to_file("micro_watersheds.geojson", driver="GeoJSON")
mws.drop(columns="geometry").to_csv("micro_watersheds.csv", index=False)
display(FileLink("micro_watersheds.geojson"), FileLink("micro_watersheds.csv"))


## Choose another table

The same response includes water, agriculture and village tables. Change `table_name` below; there is no need to download the tehsil again.


In [ ]:
display(pd.DataFrame({"Table": api_data.keys(), "Rows": [len(rows) for rows in api_data.values()]}))
table_name = "terrain"
table = pd.DataFrame(api_data[table_name])
display(table.iloc[0].to_frame("First record"))
